# AMEX Default Prediction

Interview-ready client behavior modeling project for predicting default risk on the American Express Kaggle dataset.

## Workflow
1. Data loading
2. EDA
3. Preprocessing and feature engineering
4. Baseline modeling (Logistic Regression + Gradient Boosting)
5. Evaluation (AUC, PR-AUC, decile lift)
6. Explainability and segment insights

## 1. Setup
- This notebook supports Google Colab + Google Drive as the primary data source.
- Primary files expected in Drive folder `'/content/drive/MyDrive/amex_data_parquet'`:
  - `train_data.parquet`
  - `train_labels.csv`
- Local fallback (non-Colab): `../data/raw/amex-default-prediction/`.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

RANDOM_STATE = 42
SAMPLE_FRAC = 1.0  # lower to 0.2 for quick iterations
MAX_TREND_COLS = 40
USE_GOOGLE_DRIVE_DATA = True
GOOGLE_DRIVE_DIR = Path('/content/drive/MyDrive/amex_data_parquet')
LOCAL_DATA_DIR = Path('../data/raw/amex-default-prediction')

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB and USE_GOOGLE_DRIVE_DATA:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Drive mount skipped/failed: {e}')

try:
    import kagglehub
    HAS_KAGGLEHUB = True
except Exception:
    HAS_KAGGLEHUB = False

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print({
    'in_colab': IN_COLAB,
    'kagglehub': HAS_KAGGLEHUB,
    'lightgbm': HAS_LGBM,
    'xgboost': HAS_XGB,
    'shap': HAS_SHAP
})


{'lightgbm': True, 'xgboost': True, 'shap': True}


## 2. Data Loading

In [3]:
def ensure_amex_data_dir(
    use_google_drive_data: bool = True,
    google_drive_dir: Path = GOOGLE_DRIVE_DIR,
    local_data_dir: Path = LOCAL_DATA_DIR,
) -> Path:
    if use_google_drive_data and IN_COLAB:
        train_parquet_drive = google_drive_dir / 'train_data.parquet'
        labels_drive = google_drive_dir / 'train_labels.csv'

        if train_parquet_drive.exists() and labels_drive.exists():
            print(f'Using existing Drive dataset: {google_drive_dir}')
            return google_drive_dir

        if not HAS_KAGGLEHUB:
            raise ImportError(
                'kagglehub is required in Colab when Drive files are missing. Install with: pip install kagglehub'
            )

        print('Downloading AMEX parquet dataset via kagglehub...')
        src_path = Path(kagglehub.dataset_download('ruchi798/parquet-files-amexdefault-prediction'))

        if google_drive_dir.exists():
            shutil.rmtree(google_drive_dir)
        shutil.copytree(src_path, google_drive_dir)

        print('Saved to:', google_drive_dir)
        print('Files:', os.listdir(google_drive_dir)[:10])
        return google_drive_dir

    return local_data_dir


def load_amex_raw(data_dir: Path, sample_frac: float = 1.0, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_csv = data_dir / 'train_data.csv'
    train_parquet = data_dir / 'train_data.parquet'
    labels_path = data_dir / 'train_labels.csv'

    if train_parquet.exists():
        df = pd.read_parquet(train_parquet)
    elif train_csv.exists():
        df = pd.read_csv(train_csv)
    else:
        raise FileNotFoundError(f'Expected train_data.csv or train_data.parquet in {data_dir}.')

    if not labels_path.exists():
        raise FileNotFoundError(f'Expected train_labels.csv in {data_dir}.')

    labels = pd.read_csv(labels_path)

    if sample_frac < 1.0:
        sampled_customers = (
            labels.sample(frac=sample_frac, random_state=random_state)['customer_ID']
            .drop_duplicates()
            .tolist()
        )
        df = df[df['customer_ID'].isin(sampled_customers)].copy()
        labels = labels[labels['customer_ID'].isin(sampled_customers)].copy()

    df['S_2'] = pd.to_datetime(df['S_2'])
    return df, labels

DATA_DIR = ensure_amex_data_dir(
    use_google_drive_data=USE_GOOGLE_DRIVE_DATA,
    google_drive_dir=GOOGLE_DRIVE_DIR,
    local_data_dir=LOCAL_DATA_DIR,
)

raw_df, labels_df = load_amex_raw(DATA_DIR, sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
print('Data dir:', DATA_DIR)
print(raw_df.shape, labels_df.shape)
raw_df.head(3)


KeyboardInterrupt: 

## 3. EDA (Quick but Decision-Oriented)

In [ ]:
print('Raw columns:', len(raw_df.columns))
print('Unique customers:', raw_df['customer_ID'].nunique())
print('Date range:', raw_df['S_2'].min(), 'to', raw_df['S_2'].max())

labels_df['target'].value_counts(normalize=True).rename('target_rate')

In [ ]:
# Missingness profile
missing_rate = raw_df.isna().mean().sort_values(ascending=False)
missing_rate.head(15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

labels_df['target'].value_counts().sort_index().plot(kind='bar', ax=ax[0], title='Target Count (0=Non-default, 1=Default)')
ax[0].set_xlabel('target')

obs_per_customer = raw_df.groupby('customer_ID').size()
obs_per_customer.plot(kind='hist', bins=20, ax=ax[1], title='Statements per customer')
ax[1].set_xlabel('num_statements')

plt.tight_layout()

## 4. Time-Aware Split and Leakage-Safe Feature Engineering

We aggregate per customer using only available historical statements and split by each customer's *latest statement date*.

In [ ]:
def safe_slope(values: np.ndarray) -> float:
    mask = ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    y = values[mask]
    x = np.arange(len(values))[mask]
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    if denom == 0:
        return 0.0
    return np.sum(x_centered * (y - y.mean())) / denom


def make_customer_features(df: pd.DataFrame, max_trend_cols: int = 40) -> pd.DataFrame:
    df = df.sort_values(['customer_ID', 'S_2']).copy()

    id_cols = ['customer_ID', 'S_2']
    feature_cols = [c for c in df.columns if c not in id_cols]
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    trend_cols = numeric_cols[:max_trend_cols]

    grouped = df.groupby('customer_ID', sort=False)

    pieces = []

    # Core sequence stats
    num_agg = grouped[numeric_cols].agg(['last', 'mean', 'std', 'min', 'max'])
    num_agg.columns = [f'{col}_{stat}' for col, stat in num_agg.columns]
    pieces.append(num_agg)

    # Missingness behavior
    miss = grouped[numeric_cols].apply(lambda g: g.isna().mean())
    miss.columns = [f'{c}_missing_rate' for c in miss.columns]
    pieces.append(miss)

    # Trend features for a controllable subset
    trend_frames = []
    for c in trend_cols:
        slopes = grouped[c].apply(lambda s: safe_slope(s.values.astype(float)))
        trend_frames.append(slopes.rename(f'{c}_trend'))
    if trend_frames:
        pieces.append(pd.concat(trend_frames, axis=1))

    # Recency and activity
    max_date = df['S_2'].max()
    time_feats = grouped['S_2'].agg(last_date='max', first_date='min', n_statements='count')
    time_feats['recency_days'] = (max_date - time_feats['last_date']).dt.days
    time_feats['history_days'] = (time_feats['last_date'] - time_feats['first_date']).dt.days
    pieces.append(time_feats.drop(columns=['last_date', 'first_date']))

    # Categorical-style handling: last value and diversity for known AMEX categorical columns when present
    known_cat_cols = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
    cat_cols = [c for c in known_cat_cols if c in df.columns]
    if cat_cols:
        cat_last = grouped[cat_cols].last().add_suffix('_last_cat')
        cat_nunique = grouped[cat_cols].nunique(dropna=True).add_suffix('_nunique_cat')
        pieces.extend([cat_last, cat_nunique])

    feats = pd.concat(pieces, axis=1).reset_index()
    return feats

customer_features = make_customer_features(raw_df, max_trend_cols=MAX_TREND_COLS)
customer_features = customer_features.merge(labels_df, on='customer_ID', how='inner')

print(customer_features.shape)
customer_features.head(3)

In [ ]:
# Time-aware split using customer latest statement date
last_statement = raw_df.groupby('customer_ID')['S_2'].max().rename('last_statement_date').reset_index()
model_df = customer_features.merge(last_statement, on='customer_ID', how='left').sort_values('last_statement_date')

cut_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:cut_idx].copy()
valid_df = model_df.iloc[cut_idx:].copy()

print('Train period max date:', train_df['last_statement_date'].max())
print('Valid period min date:', valid_df['last_statement_date'].min())
print('Train size:', train_df.shape, 'Valid size:', valid_df.shape)

## 5. Baseline Modeling

In [ ]:
TARGET_COL = 'target'
EXCLUDE_COLS = ['customer_ID', 'last_statement_date', TARGET_COL]
X_train = train_df.drop(columns=EXCLUDE_COLS)
y_train = train_df[TARGET_COL]
X_valid = valid_df.drop(columns=EXCLUDE_COLS)
y_valid = valid_df[TARGET_COL]

cat_cols = [c for c in X_train.columns if X_train[c].dtype == 'object']
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=False)),
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols),
    ],
    remainder='drop'
)

logreg = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
])

logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_valid)[:, 1]

auc_lr = roc_auc_score(y_valid, proba_lr)
pr_auc_lr = average_precision_score(y_valid, proba_lr)
print({'model': 'logistic_regression', 'auc': round(auc_lr, 5), 'pr_auc': round(pr_auc_lr, 5)})

In [ ]:
# Gradient boosting baseline (prefers LightGBM, then XGBoost)
boost_model_name = None
proba_boost = None
boost_importance = None

if HAS_LGBM:
    boost_model_name = 'lightgbm'
    lgb_train = X_train.copy()
    lgb_valid = X_valid.copy()

    for c in cat_cols:
        lgb_train[c] = lgb_train[c].astype('category')
        lgb_valid[c] = lgb_valid[c].astype('category')

    gbm = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.8,
        objective='binary',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    gbm.fit(lgb_train, y_train)
    proba_boost = gbm.predict_proba(lgb_valid)[:, 1]
    boost_importance = pd.Series(gbm.feature_importances_, index=lgb_train.columns)

elif HAS_XGB:
    boost_model_name = 'xgboost'
    xgb_model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=RANDOM_STATE,
        tree_method='hist',
        n_jobs=-1,
    )

    train_enc = X_train.copy()
    valid_enc = X_valid.copy()
    for c in cat_cols:
        train_enc[c] = train_enc[c].astype('category').cat.codes
        valid_enc[c] = valid_enc[c].astype('category').cat.codes

    xgb_model.fit(train_enc, y_train)
    proba_boost = xgb_model.predict_proba(valid_enc)[:, 1]
    boost_importance = pd.Series(xgb_model.feature_importances_, index=train_enc.columns)
else:
    print('No LightGBM/XGBoost installed; boosted baseline skipped.')

if proba_boost is not None:
    auc_boost = roc_auc_score(y_valid, proba_boost)
    pr_auc_boost = average_precision_score(y_valid, proba_boost)
    print({'model': boost_model_name, 'auc': round(auc_boost, 5), 'pr_auc': round(pr_auc_boost, 5)})

## 6. Evaluation: ROC-AUC, PR-AUC, and Decile Lift

In [ ]:
def decile_lift_table(y_true: pd.Series, y_score: np.ndarray) -> pd.DataFrame:
    eval_df = pd.DataFrame({'y_true': y_true.values, 'score': y_score})
    eval_df['decile'] = pd.qcut(eval_df['score'].rank(method='first'), 10, labels=False) + 1
    eval_df['decile'] = 11 - eval_df['decile']  # decile 1 = highest risk

    base_rate = eval_df['y_true'].mean()
    out = (
        eval_df.groupby('decile')
        .agg(n=('y_true', 'size'), bad_rate=('y_true', 'mean'))
        .reset_index()
        .sort_values('decile')
    )
    out['lift_vs_avg'] = out['bad_rate'] / base_rate
    return out

results = [{'model': 'logistic_regression', 'auc': auc_lr, 'pr_auc': pr_auc_lr}]
if proba_boost is not None:
    results.append({'model': boost_model_name, 'auc': auc_boost, 'pr_auc': pr_auc_boost})

results_df = pd.DataFrame(results).sort_values(['auc', 'pr_auc'], ascending=False)
results_df

In [ ]:
lift_lr = decile_lift_table(y_valid, proba_lr)
print('Logistic regression decile lift:')
display(lift_lr)

if proba_boost is not None:
    lift_boost = decile_lift_table(y_valid, proba_boost)
    print(f'{boost_model_name} decile lift:')
    display(lift_boost)

plt.figure(figsize=(8, 4))
plt.plot(lift_lr['decile'], lift_lr['lift_vs_avg'], marker='o', label='Logistic Regression')
if proba_boost is not None:
    plt.plot(lift_boost['decile'], lift_boost['lift_vs_avg'], marker='o', label=boost_model_name)
plt.axhline(1.0, linestyle='--', color='gray')
plt.title('Lift by Risk Decile (1 = Highest Predicted Risk)')
plt.xlabel('Decile')
plt.ylabel('Lift vs Average')
plt.legend()
plt.tight_layout()

## 7. Explainability

In [ ]:
if boost_importance is not None:
    top_imp = boost_importance.sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=top_imp.values, y=top_imp.index, orient='h')
    plt.title(f'Top 20 Global Feature Importances ({boost_model_name})')
    plt.tight_layout()

    display(top_imp.to_frame('importance'))

In [ ]:
if HAS_SHAP and proba_boost is not None:
    shap_sample = X_valid.sample(n=min(2000, len(X_valid)), random_state=RANDOM_STATE).copy()

    if boost_model_name == 'lightgbm':
        for c in cat_cols:
            shap_sample[c] = shap_sample[c].astype('category')
        explainer = shap.TreeExplainer(gbm)
        shap_values = explainer.shap_values(shap_sample)
        shap.summary_plot(shap_values, shap_sample, show=True)

    elif boost_model_name == 'xgboost':
        shap_sample_enc = shap_sample.copy()
        for c in cat_cols:
            shap_sample_enc[c] = shap_sample_enc[c].astype('category').cat.codes
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(shap_sample_enc)
        shap.summary_plot(shap_values, shap_sample_enc, show=True)
else:
    print('SHAP skipped (missing package or boosted model).')

## 8. Segment-Level Behavior Insights

In [ ]:
analysis_df = valid_df[['customer_ID', 'target']].copy()
analysis_df['score_lr'] = proba_lr
analysis_df['risk_decile_lr'] = pd.qcut(analysis_df['score_lr'].rank(method='first'), 10, labels=False) + 1
analysis_df['risk_decile_lr'] = 11 - analysis_df['risk_decile_lr']

key_behavior_cols = [
    c for c in valid_df.columns
    if c.endswith('_trend') or c.endswith('_std') or c.endswith('_missing_rate')
]
key_behavior_cols = key_behavior_cols[:12]

segment_view = valid_df[['customer_ID'] + key_behavior_cols].merge(
    analysis_df[['customer_ID', 'risk_decile_lr', 'target']],
    on='customer_ID', how='left'
)

segment_summary = (
    segment_view.groupby('risk_decile_lr')[key_behavior_cols + ['target']]
    .mean()
    .sort_index()
)

segment_summary.head(10)

## 9. Interview Talking Points

- Why time-aware validation matters: random splits can leak future behavior into training; customer last-statement split is safer.
- Feature design: recency, volatility (`std`), directional trend (`trend`), and missingness patterns often carry default signals.
- Baseline progression: start linear for interpretability, then use gradient boosting to capture nonlinearity and interactions.
- Business usage: decile lift supports risk-tier policying (collections prioritization, credit line review, proactive outreach).
- Next improvements: target encoding with out-of-fold strategy, sequence models, and monotonic constraints for risk governance.